# Test Tiny-GenImage Data Loader

Notebook này dùng để kiểm tra loader chung trong `data_loader/` trước khi đưa vào train Stage 1/Stage 2.

In [1]:
# Chạy cell này trên Colab/Jupyter nếu thiếu thư viện.
%pip install -q datasets torchvision matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Import loader từ repo

In [2]:
from pathlib import Path
import sys

# Nếu mở notebook từ root của repo thì giữ nguyên Path.cwd().
# Nếu chạy Colab và repo nằm trong Drive, sửa PROJECT_ROOT cho đúng.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data_loader").exists() and (PROJECT_ROOT.parent / "data_loader").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT =", PROJECT_ROOT)

from data_loader import (
    TinyGenImageDataset,
    TinyGenImageIterableDataset,
    TinyGenImageSplitConfig,
    build_image_transform,
    build_tiny_genimage_splits,
    collate_unified_batch,
)

PROJECT_ROOT = d:\LLM\LVLM\MLLM_detect_fake_image\HoangHa_Code


## 2. Chọn split cần test

In [3]:
# Các case đang hỗ trợ:
# - combined: train all generators, eval all generators
# - in_domain: train/eval real + một fake generator cụ thể
# - cross_generator: leave-one-generator-out
# - train_one_generator: train real + một fake generator, eval all generators

EVAL_CASE = "combined"
GENERATOR = "BigGAN"          # dùng cho in_domain
HELDOUT_GENERATOR = "GLIDE"   # dùng cho cross_generator
BASE_GENERATOR = "BigGAN"     # dùng cho train_one_generator
STREAMING = True               # True = đọc lazy từ Hugging Face, không tải full dataset về local cache
BALANCE_REAL = True            # True = số ảnh real bằng số ảnh fake trong từng split
CACHE_DIR = None               # chỉ dùng khi STREAMING = False; ví dụ: "/content/drive/MyDrive/hf_cache"

config = TinyGenImageSplitConfig(
    eval_case=EVAL_CASE,
    generator=GENERATOR,
    heldout_generator=HELDOUT_GENERATOR,
    base_generator=BASE_GENERATOR,
    cache_dir=CACHE_DIR,
    streaming=STREAMING,
    balance_real=BALANCE_REAL,
    seed=42,
)

splits = build_tiny_genimage_splits(config)
print(splits["notes"])
print("balance_real:", splits.get("balance_real"))
print("train real/fake:", splits.get("train_real_count"), splits.get("train_fake_count"))
print("eval real/fake:", splits.get("eval_real_count"), splits.get("eval_fake_count"))
print("train rows:", splits.get("train_rows") if STREAMING else len(splits["train"]))
print("eval rows:", splits.get("eval_rows") if STREAMING else len(splits["eval"]))
print("features:", splits["train"].features)

Train on all Tiny-GenImage train generators; evaluate on all validation generators with balanced real/fake counts.
balance_real: True
train real/fake: 14000 14000
eval real/fake: 3500 3500
train rows: 28000
eval rows: 7000
features: {'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['real', 'fake']), 'generator': ClassLabel(names=['Real', 'ADM', 'BigGAN', 'GLIDE', 'Midjourney', 'SD14', 'SD15', 'VQDM', 'Wukong'])}


## 3. Bọc bằng PyTorch Dataset/DataLoader

In [ ]:
from torch.utils.data import DataLoader

IMAGE_SIZE = 224
BATCH_SIZE = 8

train_transform = build_image_transform(image_size=IMAGE_SIZE, train=True, normalize=True)
eval_transform = build_image_transform(image_size=IMAGE_SIZE, train=False, normalize=True)

DatasetClass = TinyGenImageIterableDataset if splits.get("streaming") else TinyGenImageDataset

train_dataset = DatasetClass(
    splits["train"],
    split_name=splits["train_split_name"],
    eval_case=splits["eval_case"],
    transform=train_transform,
    task_type="classification",
)

eval_dataset = DatasetClass(
    splits["eval"],
    split_name=splits["eval_split_name"],
    eval_case=splits["eval_case"],
    transform=eval_transform,
    task_type="classification",
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False if STREAMING else True,
    num_workers=0,
    collate_fn=collate_unified_batch,
)

batch = next(iter(train_loader))
print("image tensor:", batch["image"].shape)
print("labels:", batch["label"].tolist())
print("label names:", batch["label_name"])
print("generators:", batch["generator"])
print("first metadata:", batch["metadata"][0])

'HTTPSConnectionPool(host='us.aws.cdn.hf.co', port=443): Read timed out.' thrown while requesting GET https://huggingface.co/datasets/TheKernel01/Tiny-GenImage/resolve/89c4fe9efd0ebc7ce5c7641ef57d578ccd639c69/data/train-00003-of-00014.parquet
Retrying in 1s [Retry 1/5].


## 4. Hiển thị vài ảnh trong batch

In [ ]:
import matplotlib.pyplot as plt
import torch

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denormalize(image_tensor):
    image = image_tensor.cpu() * std + mean
    return image.clamp(0, 1).permute(1, 2, 0).numpy()

num_show = min(8, batch["image"].shape[0])
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for i in range(num_show):
    axes[i].imshow(denormalize(batch["image"][i]))
    axes[i].set_title(f'{batch["label_name"][i]} | {batch["generator"][i]}')
    axes[i].axis("off")

for i in range(num_show, len(axes)):
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## 5. Test robustness transform

In [ ]:
# Ví dụ: dùng cùng eval split nhưng thêm JPEG compression ở test.
robust_transform = build_image_transform(
    image_size=IMAGE_SIZE,
    train=False,
    normalize=True,
    perturbation="jpeg",
    jpeg_quality=50,
)

robust_eval_dataset = DatasetClass(
    splits["eval"],
    split_name=splits["eval_split_name"],
    eval_case=f'{splits["eval_case"]}:jpeg_q50',
    transform=robust_transform,
    task_type="classification",
)

robust_loader = DataLoader(
    robust_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_unified_batch,
)

robust_batch = next(iter(robust_loader))
print("robust image tensor:", robust_batch["image"].shape)
print("robust eval case:", robust_batch["metadata"][0]["eval_case"])

## 6. Stage 2 alignment mode

In [ ]:
# Stage 2 sẽ cần target_text/label_token để train projector cho token real/fake.
alignment_source = splits["train"].take(8) if STREAMING else splits["train"].select(range(8))
alignment_dataset = DatasetClass(
    alignment_source,
    split_name=splits["train_split_name"],
    eval_case=splits["eval_case"],
    transform=eval_transform,
    task_type="alignment",
)
alignment_loader = DataLoader(alignment_dataset, batch_size=4, collate_fn=collate_unified_batch)
alignment_batch = next(iter(alignment_loader))
print("label_token:", alignment_batch["label_token"])
print("target_text:", alignment_batch["target_text"])